# 06 — Master Ensemble Stacking, Calibration & Submission Packaging

**Focus:** Multi-Model Stacking, Temperature Calibration ($T=0.90$), Prior Regularization ($\lambda=0.01$), and Container Packaging  
**Final Submission Package:** `submission_master_v3_lam01.zip` (Rank #50, Best Private Log Loss 0.3003, AUROC 0.9608).

---

## The Ensemble Pipeline
In the endgame, our pipeline synthesized predictions from 7 distinct model arms across 91 checkpoints:
1. `a3d_hemi`: Bilateral Siamese 3D CNN (25 checkpoints)
2. `a3d_ssl`: VICReg Self-Supervised 3D CNN (10 checkpoints)
3. `featc`: 116 Radiomics + Ridge Logistic Regression (1 checkpoint)
4. `a2d_cnx_cor`: Coronal ConvNeXt-Tiny (5 checkpoints)
5. `a2d_v2s`: Axial EfficientNetV2-S (10 checkpoints)
6. `a3d_hemisoft`: Distilled Soft-Target HemiNet (25 checkpoints)
7. `a3d_crophistsoft`: Histogram-Standardized Distilled HemiNet (15 checkpoints)

In [ ]:
import numpy as np
import pandas as pd
from scipy.special import expit as sigmoid
from scipy.special import logit
from sklearn.metrics import log_loss, roc_auc_score

# Simulate Out-of-Fold Predictions from 7 Arms
np.random.seed(42)
n_scans = 1362
y = np.array([1]*747 + [0]*615)
np.random.shuffle(y)

arms = ['a3d_hemi', 'a3d_ssl', 'featc', 'a2d_cnx_cor', 'a2d_v2s', 'a3d_hemisoft', 'a3d_crophistsoft']
oof_matrix = {}

for arm in arms:
    # Slightly correlated predictions with individual inductive biases
    noise = np.random.normal(0, 0.35, n_scans)
    signal = np.where(y == 1, 1.4, -1.4) + noise
    oof_matrix[arm] = sigmoid(signal)

df_oof = pd.DataFrame(oof_matrix)
print("OOF Predictions Matrix (First 5 Scans):")
print(df_oof.head())

In [ ]:
# Step 1: Equal Weight Stacking vs Fitted Weights
# Stacking analysis proved equal weighting (1/7 each) beat fitted simplex weights under domain shift
equal_probs = df_oof.mean(axis=1).values
equal_loss = log_loss(y, equal_probs)
equal_auc = roc_auc_score(y, equal_probs)

print(f"Raw Equal-Weight Ensemble Log Loss: {equal_loss:.4f} | AUROC: {equal_auc:.4f}")

# Step 2: Temperature Scaling (T = 0.90) and Prior Mixture Regularization (lambda = 0.01)
def calibrate_predictions(probs, T=0.90, lam=0.01, prior=0.548458):
    # Convert to log-odds
    eps = 1e-6
    probs_clamped = np.clip(probs, eps, 1 - eps)
    logits = logit(probs_clamped)
    
    # Temperature scaling
    scaled_logits = logits / T
    calibrated_probs = sigmoid(scaled_logits)
    
    # Prior mixture regularization: P_final = (1 - lambda) * P + lambda * Prior
    final_probs = (1.0 - lam) * calibrated_probs + lam * prior
    return np.clip(final_probs, 1e-4, 1.0 - 1e-4)

cal_probs = calibrate_predictions(equal_probs, T=0.90, lam=0.01)
cal_loss = log_loss(y, cal_probs)
cal_auc = roc_auc_score(y, cal_probs)

print(f"Calibrated Ensemble Log Loss (T=0.90, lam=0.01): {cal_loss:.4f} | AUROC: {cal_auc:.4f}")
print(f"Optimization Improvement: {equal_loss - cal_loss:+.4f} nats")

## Step 3: Platform Container Requirements & Self-Test Contract
DrivenData code submission requires:
- `main.py` entrypoint accepting `--input-dir` and `--output-file`.
- Evaluation completed within **2 hours** on platform CPU/GPU container.
- Per-scan isolation (zero transductive leakage).
- Deterministic output format: 2 columns (`uid`, `is_pathologic`).

The `scripts/local_e2e_v3.py` script validates these constraints before uploading.